In [ ]:

import pandas as pd
import numpy as np
import re
import string

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer



In [31]:
import pandas as pd

df = pd.read_csv("twitter_training.csv", header=None)
df.head()

,0,1,2,3
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [33]:
df.columns = ["ID", "Game", "Sentiment", "Text"]
df.head()

,ID,Game,Sentiment,Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [35]:
df.isnull().sum()

ID             0
Game           0
Sentiment      0
Text         686
dtype: int64

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         74682 non-null  int64 
 1   Game       74682 non-null  object
 2   Sentiment  74682 non-null  object
 3   Text       73996 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [39]:
missing_text_count = df['Text'].isnull().sum()
print("Number of missing text rows:", missing_text_count)
missing_text_rows = df[df['Text'].isnull()]
print(missing_text_rows)

Number of missing text rows: 686
         ID         Game Sentiment Text
61     2411  Borderlands   Neutral  NaN
553    2496  Borderlands   Neutral  NaN
589    2503  Borderlands   Neutral  NaN
745    2532  Borderlands  Positive  NaN
1105   2595  Borderlands  Positive  NaN
...     ...          ...       ...  ...
73972  9073       Nvidia  Positive  NaN
73973  9073       Nvidia  Positive  NaN
74421  9154       Nvidia  Positive  NaN
74422  9154       Nvidia  Positive  NaN
74423  9154       Nvidia  Positive  NaN

[686 rows x 4 columns]


In [41]:
# Drop rows with missing text
df = df.dropna(subset=['Text'])

# Check new dataset shape
print("Dataset shape after dropping missing text:", df.shape)

Dataset shape after dropping missing text: (73996, 4)


In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

In [45]:
# Initialize stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [47]:
# Cleaning function
def clean_text(text):
    """
    Performs full text cleaning:
    - Lowercase
    - Remove URLs
    - Remove punctuation
    - Remove numbers
    - Tokenize
    - Remove stopwords
    - Lemmatize
    """
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Tokenize
    words = text.split()
    
    # Remove stopwords and lemmatize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(words)

In [71]:
# Apply cleaning function
df.loc[:, "Clean_Text"] = df["Text"].apply(clean_text)

In [38]:
# Preview cleaned text
df[["Text", "Clean_Text"]].head()

,Text,Clean_Text
0,im getting on borderlands and i will murder yo...,im getting borderland murder
1,I am coming to the borders and I will kill you...,coming border kill
2,im getting on borderlands and i will kill you ...,im getting borderland kill
3,im coming on borderlands and i will murder you...,im coming borderland murder
4,im getting on borderlands 2 and i will murder ...,im getting borderland murder


In [53]:

# Convert text labels to numeric for classification
encoder = LabelEncoder()
df.loc[:, "Label"] = encoder.fit_transform(df["Sentiment"])

# See mapping
label_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
print("Label Mapping:", label_mapping)

y = df["Label"]

Label Mapping: {'Irrelevant': 0, 'Negative': 1, 'Neutral': 2, 'Positive': 3}


In [55]:
# Import TF-IDF vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize vectorizer
# max_features=5000 limits to top 5000 words by frequency
vectorizer = TfidfVectorizer(max_features=5000)

# Fit and transform the cleaned text
X = vectorizer.fit_transform(df["Clean_Text"])

print("TF-IDF feature matrix shape:", X.shape)

TF-IDF feature matrix shape: (73996, 5000)


In [57]:
# STEP 5: Document Similarity (Cosine)
# WARNING: Full 74k x 74k similarity matrix is too large
# Use a sample of first 1000 rows
sample_X = X[:1000]

similarity_matrix = cosine_similarity(sample_X)

# Print similarity for first 5 tweets
print("Cosine Similarity (first 5 tweets):\n", similarity_matrix[:5, :5])

Cosine Similarity (first 5 tweets):
 [[1.         0.         0.58531629 0.76696981 1.        ]
 [0.         1.         0.29073809 0.26604087 0.        ]
 [0.58531629 0.29073809 1.         0.34340017 0.58531629]
 [0.76696981 0.26604087 0.34340017 1.         0.76696981]
 [1.         0.         0.58531629 0.76696981 1.        ]]


In [61]:

# STEP 6: Clustering (K-Means)
# Cluster all data into 3 clusters (example)
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X)

# Assign cluster labels
df.loc[:, "Cluster"] = kmeans.labels_

# Compare clusters with sentiment
cluster_vs_sentiment = pd.crosstab(df["Cluster"], df["Sentiment"])
print("Cluster vs Sentiment:\n", cluster_vs_sentiment)

Cluster vs Sentiment:
 Sentiment  Irrelevant  Negative  Neutral  Positive
Cluster                                           
0               11345     18283    16194     17367
1                 680       586      647       911
2                 850      3489     1267      2377


In [63]:
# STEP 7: Text Classification
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 1. Naive Bayes -----
nb = MultinomialNB()
nb.fit(X_train, y_train)
nb_pred = nb.predict(X_test)
nb_acc = accuracy_score(y_test, nb_pred)
print("Naive Bayes Accuracy:", nb_acc)


Naive Bayes Accuracy: 0.6316891891891891


In [65]:

#  2. Logistic Regression -----
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)
print("Logistic Regression Accuracy:", lr_acc)



Logistic Regression Accuracy: 0.6835135135135135


In [67]:
# 3. Support Vector Machine 
svm = LinearSVC()
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)
svm_acc = accuracy_score(y_test, svm_pred)
print("SVM Accuracy:", svm_acc)



SVM Accuracy: 0.7056081081081081


In [69]:
# Compare models
print("\nModel Comparison:")
print(f"Naive Bayes: {nb_acc:.4f}")
print(f"Logistic Regression: {lr_acc:.4f}")
print(f"SVM: {svm_acc:.4f}")


Model Comparison:
Naive Bayes: 0.6317
Logistic Regression: 0.6835
SVM: 0.7056
